In [5]:
import os
import base64
import requests
import pandas as pd
from urllib.parse import urlparse
from dotenv import load_dotenv
from time import sleep

# === Load GitHub Tokens for Rotation ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]

if not tokens:
    raise ValueError("❌ No GitHub tokens found.")

token_index = 0

def get_headers():
    return {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-checker"
    }

def rotate_token():
    global token_index
    token_index = (token_index + 1) % len(tokens)
    print(f"🔁 Rotated to token #{token_index + 1}")

def safe_get(url, special_headers=None, max_retries=5):
    global token_index
    for _ in range(max_retries):
        headers = get_headers()
        if special_headers:
            headers.update(special_headers)
        r = requests.get(url, headers=headers)
        if r.status_code == 200:
            return r
        elif r.status_code == 403:
            rotate_token()
            sleep(2)
        else:
            sleep(1)
    return None

def extract_repo_info(url):
    parts = urlparse(url).path.strip("/").split("/")
    if len(parts) >= 2:
        return parts[0], parts[1]
    return None, None

# === Load Input CSV ===
input_path = r"C:\\Android Mobile App\\Step1_URL_Search\\1-LanguageThenAndroid-Stepwise_ API_Check_only\\Has_Activity_Only_Keyword.csv"
df = pd.read_csv(input_path)

results = []

# === Analyze Each Repo ===
for idx, url in enumerate(df['html_url'], start=1):
    print(f"[{idx}/{len(df)}] Checking repo: {url}")
    owner, repo = extract_repo_info(url)
    if not owner or not repo:
        continue

    base_url = f"https://api.github.com/repos/{owner}/{repo}"

    try:
        # --- Repo Metadata ---
        r = safe_get(base_url)
        if not r:
            raise Exception("Repo metadata fetch failed")
        repo_data = r.json()

        # --- Repo Topics ---
        topic_url = f"{base_url}/topics"
        r = safe_get(topic_url, special_headers={"Accept": "application/vnd.github.mercy-preview+json"})
        if not r:
            raise Exception("Repo topics fetch failed")
        topics = r.json().get("names", [])

        # --- README Content ---
        readme_url = f"{base_url}/readme"
        r = safe_get(readme_url)

        readme_text = ""
        if r and r.status_code == 200:
            content = r.json().get("content", "")
            readme_text = base64.b64decode(content).decode("utf-8", errors="ignore")

        # --- Checks ---
        android_in_topic = "android" in topics
        android_in_name_or_desc = "android" in f"{repo_data.get('name', '')} {repo_data.get('description', '')}".lower()
        android_in_readme = "android" in readme_text.lower()
        language = repo_data.get("language", "")

        is_android_repo = android_in_topic or android_in_name_or_desc or android_in_readme

        results.append({
            "repo_url": url,
            "is_android_repo": is_android_repo,
            "language": language,
            "android_in_topic": android_in_topic,
            "android_in_name_or_desc": android_in_name_or_desc,
            "android_in_readme": android_in_readme
        })

    except Exception as e:
        results.append({
            "repo_url": url,
            "error": str(e)
        })

# === Save Output ===
output_df = pd.DataFrame(results)
output_path = r"C:\\Android Mobile App\\Step1_URL_Search\\1-LanguageThenAndroid-Stepwise_ API_Check_only\\Has_Activity_Only_Keyword_metadata.csv"
output_df.to_csv(output_path, index=False)

print(f"✅ Review complete. Output saved to:\n{output_path}")


[1/3797] Checking repo: https://github.com/0015/ThatProject
🔁 Rotated to token #2
[2/3797] Checking repo: https://github.com/0x7c13/Pal3.Unity
[3/3797] Checking repo: https://github.com/0xkol/badspin
[4/3797] Checking repo: https://github.com/12-10-8/ncnn-android-pose
[5/3797] Checking repo: https://github.com/1280103995/react-native-elm
[6/3797] Checking repo: https://github.com/1c7/CrashCourse-Android-App
[7/3797] Checking repo: https://github.com/1hbb/react-native-instagram-clone
[8/3797] Checking repo: https://github.com/1payment/wallet
[9/3797] Checking repo: https://github.com/20000s/android-detector
[10/3797] Checking repo: https://github.com/2003scape/rsc-c
[11/3797] Checking repo: https://github.com/219-design/qt-qml-project-template-with-ci
[12/3797] Checking repo: https://github.com/2468785842/krkr2
[13/3797] Checking repo: https://github.com/29ki/29k
[14/3797] Checking repo: https://github.com/2gis/qtandroidextensions
[15/3797] Checking repo: https://github.com/378056350/bo